<a href="https://colab.research.google.com/github/apirakqqqqq/GE337_Programming/blob/main/Lab_4/Lab_4_6606614870_%E0%B8%AD%E0%B8%A0%E0%B8%B4%E0%B8%A3%E0%B8%B1%E0%B8%81%E0%B8%A9%E0%B9%8C_%E0%B8%9B%E0%B8%B1%E0%B8%8D%E0%B8%8D%E0%B8%B2%E0%B8%AA%E0%B8%B2%E0%B8%84%E0%B8%A3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install geopandas rasterio folium shapely matplotlib -q

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**โหลดข้อมูล**

In [50]:
import rasterio

raster_path = "/content/S2_BKK_7Bands.tif"

src = rasterio.open(raster_path)

print("===== Metadata =====")
print(src.meta)

print("CRS:", src.crs)
print("Width:", src.width)
print("Height:", src.height)
print("Number of bands:", src.count)

===== Metadata =====
{'driver': 'GTiff', 'dtype': 'float32', 'nodata': None, 'width': 3340, 'height': 3898, 'count': 7, 'crs': CRS.from_wkt('GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]'), 'transform': Affine(0.0001796630568239043, 0.0, 100.29995574391057,
       0.0, -0.0001796630568239043, 14.100316025653656)}
CRS: EPSG:4326
Width: 3340
Height: 3898
Number of bands: 7


In [51]:
import pandas as pd

points = pd.read_csv("/content/bkk_training_points.csv")

print("===== Training Points =====")
print(points.head())

print("\nClass distribution")
print(points["class"].value_counts())

===== Training Points =====
         lat         lon                                            name  \
0  13.752703  100.489817                                 ็ต้นไม้รูปหัวใจ   
1  13.568089  100.566689                    สนามกีฬา อำเภอพระสมุทรเจดีย์   
2  13.633630  100.611424                                             NaN   
3  13.830157  100.538337  Prachanukun Health Garden สวนสุขภาพ ประชานุกูล   
4  13.727300  100.580341                                 Angoon's Garden   

   class  
0  green  
1  green  
2  green  
3  green  
4  green  

Class distribution
class
water    200
urban    142
green     80
Name: count, dtype: int64


ดึง BAND + Raster

In [52]:
coords = [(x,y) for x,y in zip(points["lon"], points["lat"])]

In [53]:
samples = []

for val in src.sample(coords):
    samples.append(val)

In [54]:
band_names = [f"band{i+1}" for i in range(src.count)]

features = pd.DataFrame(samples, columns=band_names)

print(features.head())

    band1   band2   band3   band4   band5   band6     band7
0  1534.0  1790.0  2048.5  2606.5  3067.5  2653.0  0.119871
1   756.0  1240.0  1177.0  3953.0  3798.0  2309.0  0.541131
2   704.0  1093.0  1020.5  2855.0  2237.0  1703.5  0.473358
3   338.5   528.0   423.0  2516.0  1781.5  1171.5  0.712147
4   453.0   712.0   626.5  2937.0  2231.0  1538.5  0.648379


**คำนวณ NDVI**

In [55]:
features["ndvi"] = (features["band4"] - features["band3"]) / (features["band4"] + features["band3"])

**รวม Raster + Vector เป็น DataFrame**

In [56]:
df = pd.concat([points.reset_index(drop=True), features], axis=1)

print("===== Combined Data =====")
print(df.head())

===== Combined Data =====
         lat         lon                                            name  \
0  13.752703  100.489817                                 ็ต้นไม้รูปหัวใจ   
1  13.568089  100.566689                    สนามกีฬา อำเภอพระสมุทรเจดีย์   
2  13.633630  100.611424                                             NaN   
3  13.830157  100.538337  Prachanukun Health Garden สวนสุขภาพ ประชานุกูล   
4  13.727300  100.580341                                 Angoon's Garden   

   class   band1   band2   band3   band4   band5   band6     band7      ndvi  
0  green  1534.0  1790.0  2048.5  2606.5  3067.5  2653.0  0.119871  0.119871  
1  green   756.0  1240.0  1177.0  3953.0  3798.0  2309.0  0.541131  0.541131  
2  green   704.0  1093.0  1020.5  2855.0  2237.0  1703.5  0.473358  0.473358  
3  green   338.5   528.0   423.0  2516.0  1781.5  1171.5  0.712147  0.712147  
4  green   453.0   712.0   626.5  2937.0  2231.0  1538.5  0.648379  0.648379  


**แบ่ง Training / Testing**

In [57]:
from sklearn.model_selection import train_test_split

feature_cols = band_names + ["ndvi"]

X = df[feature_cols]
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("Train size:",len(X_train))
print("Test size:",len(X_test))

Train size: 295
Test size: 127


**ใช้ Random Forest จำแนกพื้นที่**

In [58]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**ปรับขนาดข้อมูล + Feature Selection**

In [38]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**ใช้ Random Forest จำแนกพื้นที่**

In [59]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)

rf.fit(X_train_scaled, y_train)

RandomForestClassifier(random_state=42)

**ใช้ Grid Search ปรับพารามิเตอร์**

In [60]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators":[50,100,200],
    "max_depth":[5,10,20],
    "min_samples_split":[2,5]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    n_jobs=-1
)

grid.fit(X_train_scaled, y_train)

best_model = grid.best_estimator_

print("Best Parameters:",grid.best_params_)

Best Parameters: {'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 50}


In [61]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = best_model.predict(X_test_scaled)

print("Accuracy:",accuracy_score(y_test,y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test,y_pred))

print("\nClassification Report")
print(classification_report(y_test,y_pred))

Accuracy: 0.7086614173228346

Confusion Matrix
[[ 9 11  4]
 [ 4 30  9]
 [ 0  9 51]]

Classification Report
              precision    recall  f1-score   support

       green       0.69      0.38      0.49        24
       urban       0.60      0.70      0.65        43
       water       0.80      0.85      0.82        60

    accuracy                           0.71       127
   macro avg       0.70      0.64      0.65       127
weighted avg       0.71      0.71      0.70       127

